Building a Context-Aware AI Chatbot using NLP Techniques.

In [185]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

# Standard libraries
import json
import pickle
import sqlite3
import random
import re
from datetime import datetime
from typing import Dict, List, Tuple, Optional

# Data handling
import pandas as pd
import numpy as np

# NLP Libraries
import nltk
import spacy
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Deep Learning (Bonus)
import torch
from transformers import pipeline

# Voice (Bonus)
import speech_recognition as sr
import pyttsx3

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

# Load spaCy model (Bonus 1)
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [186]:
# ============================================================
# CELL 2: CREATE INTENT DATASET 
# ============================================================

def create_intent_dataset() -> Dict:
    """
    Create the basic intent dataset as per project requirements.
    
    Returns:
        Dict: Dictionary containing intents with their patterns
    """
    intents = {
        "greeting": ["hi", "hello", "hey", "good morning", "good afternoon", "good evening"],
        "goodbye": ["bye", "goodbye", "see you", "see you later", "take care"],
        "weather": ["weather today", "what's the weather", "is it hot", "temperature", "forecast"],
        "name": ["what is your name", "who are you", "tell me your name", "your name please"]
    }
    
    print("✅ Intent dataset created")
    for intent, patterns in intents.items():
        print(f"   {intent}: {len(patterns)} patterns")
    
    return intents

# Create dataset
intents = create_intent_dataset()

✅ Intent dataset created
   greeting: 6 patterns
   goodbye: 5 patterns
   weather: 5 patterns
   name: 4 patterns


In [187]:
# ============================================================
# CELL 3: TEXT PREPROCESSING 
# ============================================================

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str) -> str:
    """
    Preprocess text using NLTK tokenization and lemmatization.
    
    Steps:
    1. Convert to lowercase
    2. Tokenize
    3. Lemmatize each token
    4. Join back to string
    
    Args:
        text: Raw input text
        
    Returns:
        Processed text string
    """
    if not text or not isinstance(text, str):
        return ""
    
    # Convert to lowercase and tokenize
    tokens = word_tokenize(text.lower())
    
    # Lemmatize and remove non-alphanumeric tokens
    tokens = [lemmatizer.lemmatize(token) for token in tokens if token.isalnum()]
    
    return " ".join(tokens)

# Test preprocessing
test_text = "Hello! How are you doing today?"
print(f"Original: {test_text}")
print(f"Processed: {preprocess_text(test_text)}")

Original: Hello! How are you doing today?
Processed: hello how are you doing today


In [188]:
# ============================================================
# CELL 4: FEATURE EXTRACTION & TRAIN CLASSIFIER 
# ============================================================

def prepare_training_data(intents: Dict) -> Tuple[List[str], List[str]]:
    """
    Prepare training data from intent patterns.
    
    Args:
        intents: Dictionary of intents with patterns
        
    Returns:
        Tuple of (sentences list, labels list)
    """
    sentences = []
    labels = []
    
    for intent_name, patterns in intents.items():
        for pattern in patterns:
            sentences.append(preprocess_text(pattern))
            labels.append(intent_name)
    
    return sentences, labels

def train_intent_classifier(sentences: List[str], labels: List[str]) -> Tuple[TfidfVectorizer, LogisticRegression]:
    """
    Train TF-IDF vectorizer and Logistic Regression classifier.
    
    Args:
        sentences: List of preprocessed sentences
        labels: List of corresponding intent labels
        
    Returns:
        Tuple of (vectorizer, classifier)
    """
    # Create TF-IDF vectorizer
    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    
    # Transform sentences to vectors
    X = vectorizer.fit_transform(sentences)
    
    # Train classifier
    classifier = LogisticRegression(max_iter=1000, random_state=42)
    classifier.fit(X, labels)
    
    return vectorizer, classifier

# Prepare and train
sentences, labels = prepare_training_data(intents)
vectorizer, classifier = train_intent_classifier(sentences, labels)

print(f"✅ Training complete!")
print(f"   Total training samples: {len(sentences)}")
print(f"   Unique intents: {set(labels)}")

✅ Training complete!
   Total training samples: 20
   Unique intents: {'name', 'goodbye', 'weather', 'greeting'}


In [189]:
# ============================================================
# CELL 5: RESPONSE SYSTEM 
# ============================================================

def create_response_system() -> Dict:
    """
    Create response dictionary for each intent.
    
    Returns:
        Dictionary mapping intents to response strings
    """
    responses = {
        "greeting": "Hello! How can I help you today?",
        "goodbye": "Goodbye! Have a nice day!",
        "weather": "Please tell me your city.",
        "name": "I am your AI assistant. You can call me Nexus."
    }
    
    return responses

responses = create_response_system()
print("✅ Response system created")
for intent, response in responses.items():
    print(f"   {intent}: {response}")

✅ Response system created
   greeting: Hello! How can I help you today?
   goodbye: Goodbye! Have a nice day!
   weather: Please tell me your city.
   name: I am your AI assistant. You can call me Nexus.


In [190]:
# ============================================================
# CELL 6: CONTEXT MANAGEMENT 
# ============================================================

# Global context storage
context_store = {}

def update_context(user_id: str, intent: str) -> None:
    """
    Update conversation context for a user.
    
    Args:
        user_id: Unique identifier for the user
        intent: Current intent to store
    """
    context_store[user_id] = {
        "last_intent": intent,
        "timestamp": datetime.now().isoformat()
    }

def get_context(user_id: str) -> Optional[str]:
    """
    Get last intent from context for a user.
    
    Args:
        user_id: Unique identifier for the user
        
    Returns:
        Last intent or None if not found
    """
    context = context_store.get(user_id, {})
    return context.get("last_intent")

def clear_context(user_id: str) -> None:
    """
    Clear context for a user.
    
    Args:
        user_id: Unique identifier for the user
    """
    if user_id in context_store:
        del context_store[user_id]

print("✅ Context management system ready")

✅ Context management system ready


In [191]:
# ============================================================
# CELL 7: NAMED ENTITY RECOGNITION 
# ============================================================

def extract_city(text: str) -> Optional[str]:
    """
    Extract city name from text using spaCy NER.
    
    Args:
        text: User input text
        
    Returns:
        City name if found, None otherwise
    """
    if not text:
        return None
    
    doc = nlp(text)
    
    # Extract GPE (Geo-Political Entity) entities
    for ent in doc.ents:
        if ent.label_ == "GPE":
            return ent.text
    
    # Fallback: check for common city names
    common_cities = ['london', 'karachi', 'new york', 'paris', 'tokyo', 'dubai']
    words = text.lower().split()
    for word in words:
        if word in common_cities:
            return word.title()
    
    return None

# Test NER
test_text = "I want weather in London tomorrow"
city = extract_city(test_text)
print(f"Text: {test_text}")
print(f"Extracted city: {city}")

Text: I want weather in London tomorrow
Extracted city: London


In [192]:
# ============================================================
# CELL 8: PRETRAINED MODEL 
# ============================================================

def load_bert_classifier():
    """
    Load pretrained BERT model for zero-shot classification.
    
    Returns:
        Zero-shot classification pipeline
    """
    print("Loading BERT model (this may take a moment)...")
    classifier = pipeline(
        "zero-shot-classification",
        model="typeform/distilbert-base-uncased-mnli",
        device=-1  # Use CPU
    )
    print("✅ BERT model loaded!")
    return classifier

# Load BERT (uncomment to use)
# bert_classifier = load_bert_classifier()

def predict_intent_bert(text: str, candidate_intents: List[str]) -> Tuple[str, float]:
    """
    Predict intent using pretrained BERT model.
    
    Args:
        text: User input text
        candidate_intents: List of possible intents
        
    Returns:
        Tuple of (predicted intent, confidence score)
    """
    result = bert_classifier(text, candidate_intents)
    return result['labels'][0], result['scores'][0]

print("✅ BERT integration ready (uncomment to use)")

✅ BERT integration ready (uncomment to use)


In [193]:
# ============================================================
# CELL 9: MAIN CHATBOT FUNCTION
# ============================================================

def predict_intent(user_input: str) -> str:
    """
    Predict intent using trained classifier.
    
    Args:
        user_input: User's message
        
    Returns:
        Predicted intent
    """
    processed = preprocess_text(user_input)
    vec = vectorizer.transform([processed])
    intent = classifier.predict(vec)[0]
    return intent

def chatbot(user_input: str, user_id: str = "user1", use_bert: bool = False) -> str:
    """
    Main chatbot function with context awareness.
    
    Args:
        user_input: User's message
        user_id: Unique user identifier for context
        use_bert: Whether to use BERT for intent prediction (Bonus)
        
    Returns:
        Bot response
    """
    if not user_input or not user_input.strip():
        return "Please say something."
    
    # Get last intent from context
    last_intent = get_context(user_id)
    
    # Context-aware logic: User responded to weather query
    if last_intent == "weather":
        city = extract_city(user_input)
        if city:
            update_context(user_id, "weather_done")
            return f"Weather info for {city} coming soon... 🌤️"
        elif len(user_input.split()) <= 3:
            update_context(user_id, "weather_done")
            return f"Weather info for {user_input} coming soon... 🌤️"
    
    # Predict intent
    if use_bert and 'bert_classifier' in globals():
        candidate_intents = list(responses.keys())
        intent, confidence = predict_intent_bert(user_input, candidate_intents)
    else:
        intent = predict_intent(user_input)
    
    # Update context
    update_context(user_id, intent)
    
    # Return response
    return responses.get(intent, "Sorry, I didn't understand that.")

In [194]:
# ============================================================
# CELL 10: DATABASE STORAGE 
# ============================================================

def init_database(db_path: str = "chat_history.db") -> None:
    """
    Initialize SQLite database for conversation storage.
    
    Args:
        db_path: Path to database file
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id TEXT,
            user_message TEXT,
            bot_response TEXT,
            intent TEXT,
            timestamp TEXT
        )
    ''')
    
    conn.commit()
    conn.close()
    print(f"✅ Database initialized at {db_path}")

def save_conversation(user_id: str, user_msg: str, bot_response: str, intent: str) -> None:
    """
    Save conversation to database.
    
    Args:
        user_id: User identifier
        user_msg: User's message
        bot_response: Bot's response
        intent: Predicted intent
    """
    conn = sqlite3.connect('chat_history.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        INSERT INTO conversations (user_id, user_message, bot_response, intent, timestamp)
        VALUES (?, ?, ?, ?, ?)
    ''', (user_id, user_msg, bot_response, intent, datetime.now().isoformat()))
    
    conn.commit()
    conn.close()

def get_chat_history(user_id: str, limit: int = 10) -> List[Tuple]:
    """
    Retrieve chat history for a user.
    
    Args:
        user_id: User identifier
        limit: Maximum number of records to return
        
    Returns:
        List of conversation records
    """
    conn = sqlite3.connect('chat_history.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT user_message, bot_response, intent, timestamp
        FROM conversations
        WHERE user_id = ?
        ORDER BY timestamp DESC
        LIMIT ?
    ''', (user_id, limit))
    
    history = cursor.fetchall()
    conn.close()
    return history

# Initialize database
init_database()

✅ Database initialized at chat_history.db


In [195]:
# ============================================================
# CELL 11: VOICE INPUT/OUTPUT
# ============================================================

def get_voice_input() -> Optional[str]:
    """
    Get voice input from microphone.
    
    Returns:
        Recognized text or None if failed
    """
    recognizer = sr.Recognizer()
    
    with sr.Microphone() as source:
        print("🎤 Listening... Speak now:")
        try:
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
            text = recognizer.recognize_google(audio)
            print(f"📝 Recognized: {text}")
            return text
        except sr.WaitTimeoutError:
            print("⏰ No speech detected")
            return None
        except sr.UnknownValueError:
            print("❌ Could not understand audio")
            return None
        except Exception as e:
            print(f"❌ Error: {e}")
            return None

def text_to_speech(text: str) -> None:
    """
    Convert text to speech output.
    
    Args:
        text: Text to speak
    """
    try:
        engine = pyttsx3.init()
        engine.say(text)
        engine.runAndWait()
    except Exception as e:
        print(f"❌ Speech error: {e}")

def voice_chat_session(user_id: str = "voice_user") -> None:
    """
    Run a voice-based chat session.
    
    Args:
        user_id: User identifier for context
    """
    print("\n" + "="*50)
    print("🎙️ VOICE CHAT MODE")
    print("="*50)
    print("Say 'exit' to quit")
    print("Say 'clear' to clear context")
    print("="*50 + "\n")
    
    while True:
        user_input = get_voice_input()
        
        if user_input is None:
            continue
        
        if user_input.lower() == 'exit':
            response = "Goodbye! Have a great day!"
            print(f"Bot: {response}")
            text_to_speech(response)
            break
        
        if user_input.lower() == 'clear':
            clear_context(user_id)
            response = "Conversation context cleared!"
            print(f"Bot: {response}")
            text_to_speech(response)
            continue
        
        # Get chatbot response
        response = chatbot(user_input, user_id, use_bert=False)
        print(f"Bot: {response}")
        text_to_speech(response)

print("✅ Voice functions ready (uncomment to use)")

✅ Voice functions ready (uncomment to use)


In [196]:
# ============================================================
# CELL 12: CONSOLE CHAT 
# ============================================================

def console_chat():
    """
    Run chatbot in interactive console mode.
    """
    print("\n" + "="*60)
    print("🤖 CONTEXT-AWARE CHATBOT")
    print("="*60)
    print("Commands:")
    print("  - Type your message to chat")
    print("  - Type 'exit' to quit")
    print("  - Type 'clear' to clear context")
    print("  - Type 'voice' to switch to voice mode")
    print("="*60 + "\n")
    
    user_id = "console_user"
    
    while True:
        try:
            user_input = input("You: ").strip()
            
            if user_input.lower() == 'exit':
                print("Bot: Goodbye! Have a great day!")
                break
            
            if user_input.lower() == 'clear':
                clear_context(user_id)
                print("Bot: Conversation context cleared!")
                continue
            
            if user_input.lower() == 'voice':
                voice_chat_session(user_id)
                continue
            
            if not user_input:
                continue
            
            # Get response
            response = chatbot(user_input, user_id, use_bert=False)
            print(f"Bot: {response}\n")
            
            # Save to database
            intent = predict_intent(user_input)
            save_conversation(user_id, user_input, response, intent)
            
        except KeyboardInterrupt:
            print("\nBot: Goodbye!")
            break
        except Exception as e:
            print(f"Error: {e}\n")

# Run the chatbot
console_chat()


🤖 CONTEXT-AWARE CHATBOT
Commands:
  - Type your message to chat
  - Type 'exit' to quit
  - Type 'clear' to clear context
  - Type 'voice' to switch to voice mode

Bot: Goodbye! Have a great day!


In [197]:
# ============================================================
# CELL 13: SAVE MODELS FOR WEB APP
# ============================================================

import os

def save_models():
    """
    Save trained models for use in Streamlit web app.
    """
    os.makedirs("models", exist_ok=True)
    
    # Save vectorizer
    with open("models/vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer, f)
    
    # Save classifier
    with open("models/classifier.pkl", "wb") as f:
        pickle.dump(classifier, f)
    
    # Save responses
    with open("models/responses.pkl", "wb") as f:
        pickle.dump(responses, f)
    
    # Save intents
    with open("models/intents.pkl", "wb") as f:
        pickle.dump(intents, f)
    
    print("✅ Models saved to 'models/' directory")

# Save models
save_models()

✅ Models saved to 'models/' directory
